# Elastic recoil spectra

This notebook combines

- halo models,
- scattering kinematics,
- nuclear form factors,
- interaction cross sections,

to compute observable recoil spectra.

The examples compare several target materials commonly used in direct-detection experiments.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.nuclei.isotopes import Xe131, Ar40, Si28, Ge73
from src.halo import StandardHaloModel
from src.interactions.si import SIInteraction
from src.interactions.particles import WIMP
from src.nuclei.form_factors import HelmFormFactor
from src.detectors.detectors import IdealDetector
from src.rates import RateCalculator
from src.utils.factory import create_kinematics

from src.plotting.plots_rates import plot_integral_rate

import numpy as np
import matplotlib.pyplot as plt

## Integrated event rate

The total expected number of events depends strongly on the detector target.

Heavy nuclei benefit from coherent enhancement approximately proportional to
$
A^2,
$

while light nuclei suffer less form-factor suppression and may be advantageous at larger recoil energy.

In [ ]:
Egrid = np.linspace(1, 1e5, 100)

shm = StandardHaloModel(v0=220, vesc=600).boost(vboost=240)
wimp=WIMP(mass=100e9, spin=0)

targets = [Xe131, Ge73, Ar40, Si28]
rate_calcs = []

for nuc in targets:

    kinematics = create_kinematics(wimp=wimp, nucleus=nuc)
    helm = HelmFormFactor(nucleus=nuc)
    si = SIInteraction(wimp=wimp, nucleus=nuc,
                       kinematics=kinematics,
                       form_factor=helm)
    
    detector = IdealDetector(nucleus=nuc, mass_kg=1,
                             exposure_days=365)

    rate_calc = RateCalculator(wimp=wimp, halo=shm,
                               interaction=si, detector=detector)
    rate_calcs.append(rate_calc)

fig, ax = plot_integral_rate(rate_calc=rate_calcs[0], ER=Egrid, label=r"$^{131}$Xe", 
                             ylim=(1e-3, 1))
fig, ax = plot_integral_rate(rate_calc=rate_calcs[1], ER=Egrid, label=r"$^{73}$Ge",
                             ylim=(1e-3,1), ax=ax)
fig, ax = plot_integral_rate(rate_calc=rate_calcs[2], ER=Egrid, label=r"$^{40}$Ar",
                             ylim=(1e-3,1), ax=ax)
fig, ax = plot_integral_rate(rate_calc=rate_calcs[3], ER=Egrid, label=r"$^{28}$Si",
                             ylim=(1e-3,1), ax=ax)
ax.set_ylabel("integral rate, counts/kg/year")
ax.legend()
plt.show()